# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
# 1. Corpus de Project Gutenberg: mismos 100 libros y misma tokenización que 03_TF_DF.ipynb
import math, re, tarfile, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

ARCHIVE = Path("data/txt-files.tar.zip")
N_DOCUMENTOS = 100

def tokenize(text):
    # Minúsculas; conserva letras Unicode y apóstrofes internos.
    return re.findall(r"[^\W\d_]+(?:['’][^\W\d_]+)?", text.lower(), flags=re.UNICODE)

def load_corpus(path, limit=100):
    raw, tokens = {}, {}
    if not Path(path).is_file():
        raise FileNotFoundError(f"No se encontró {path}. Abra el notebook desde la carpeta del proyecto.")
    with zipfile.ZipFile(path) as zf:
        tar_names = [n for n in zf.namelist() if n.lower().endswith('.tar')]
        if not tar_names:
            raise ValueError("El ZIP no contiene el TAR de Gutenberg esperado.")
        with zf.open(tar_names[0]) as packed, tarfile.open(fileobj=packed, mode='r|*') as tf:
            seen = set()
            for member in tf:
                if not member.isfile() or not member.name.lower().endswith('.txt'):
                    continue
                match = re.search(r"(?:^|/)pg(\d+)\.txt$", member.name, re.I)
                if not match or match.group(1) in seen:
                    continue
                stream = tf.extractfile(member)
                if stream is None:
                    continue
                doc_id = f"pg{match.group(1)}.txt"
                text = stream.read().decode('utf-8', errors='replace')
                raw[doc_id] = text
                tokens[doc_id] = tokenize(text)
                seen.add(match.group(1))
                if len(raw) >= limit:
                    break
    if len(raw) != limit:
        raise ValueError(f"Se esperaban {limit} libros; se encontraron {len(raw)}")
    return raw, tokens

def extract_title(text, doc_id):
    for line in text.splitlines()[:80]:
        match = re.match(r"\s*Title:\s*(.+)", line, re.I)
        if match:
            return match.group(1).strip()
    return doc_id

raw_texts, documents = load_corpus(ARCHIVE, N_DOCUMENTOS)
doc_ids = list(documents)
titles = {d: extract_title(raw_texts[d], d) for d in doc_ids}
tf_counts = {d: Counter(documents[d]) for d in doc_ids}
lengths = {d: len(documents[d]) for d in doc_ids}
N = len(doc_ids)
df = Counter()
for counts in tf_counts.values():
    df.update(counts.keys())
vocabulary = sorted(df)

# Tabla solicitada: ID | título | texto | tokens; se conserva texto original.
tabla_corpus = pd.DataFrame([
    {"Identificador de documento": d, "Título": titles[d], "Texto": raw_texts[d], "tokens": documents[d]}
    for d in doc_ids
])
print(f"Documentos: {N:,} | Vocabulario: {len(vocabulary):,} | Tokens: {sum(lengths.values()):,}")
print(f"Longitud de documento: min={min(lengths.values()):,}, media={np.mean(list(lengths.values())):,.1f}, max={max(lengths.values()):,}")
display(tabla_corpus[["Identificador de documento", "Título", "Texto", "tokens"]].head())

# DF e IDF suave, consistente con la práctica 03.
idf = {term: math.log(N / df[term]) + 1 for term in vocabulary}
tabla_df_idf = pd.DataFrame([{"Término": t, "DF": df[t], "IDF suave ln(N/DF)+1": idf[t]} for t in vocabulary])
print("DF e IDF de muestra (10 términos más frecuentes y 10 más raros presentes en >=2 documentos):")
frecuentes = sorted(vocabulary, key=lambda t: (-df[t], t))[:10]
raros = sorted((t for t in vocabulary if df[t] >= 2), key=lambda t: (df[t], t))[:10]
display(tabla_df_idf[tabla_df_idf["Término"].isin(frecuentes + raros)].sort_values(["DF", "Término"]))

Documentos: 100 | Vocabulario: 113,070 | Tokens: 5,669,041
Longitud de documento: min=5,251, media=56,690.4, max=180,935


,Identificador de documento,Título,Texto,tokens
0,pg10000.txt,The Magna Carta,The Project Gutenberg eBook of The Magna Carta...,"[the, project, gutenberg, ebook, of, the, magn..."
1,pg10001.txt,Apocolocyntosis,The Project Gutenberg eBook of Apocolocyntosis...,"[the, project, gutenberg, ebook, of, apocolocy..."
2,pg10002.txt,The House on the Borderland,The Project Gutenberg eBook of The House on th...,"[the, project, gutenberg, ebook, of, the, hous..."
3,pg10003.txt,"My First Years as a Frenchwoman, 1876-1879",The Project Gutenberg eBook of My First Years ...,"[the, project, gutenberg, ebook, of, my, first..."
4,pg10004.txt,The Warriors,The Project Gutenberg eBook of The Warriors\r\...,"[the, project, gutenberg, ebook, of, the, warr..."


DF e IDF de muestra (10 términos más frecuentes y 10 más raros presentes en >=2 documentos):


,Término,DF,IDF suave ln(N/DF)+1
4,a'mighty,2,4.912023
5,a'n,2,4.912023
9,a's,2,4.912023
29,abandonest,2,4.912023
39,abandon’st,2,4.912023
42,abased,2,4.912023
49,abash’d,2,4.912023
58,abateth,2,4.912023
59,abati,2,4.912023
79,abbatte,2,4.912023


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
# 2. Ocho consultas inglesas: frecuentes, poco frecuentes y de distintas longitudes.
consultas = {
    "Q1 — comunes, breve": {"texto": "love time", "intencion": "Buscar pasajes sobre amor y tiempo.", "df_objetivo": ["love", "time"]},
    "Q2 — ballenas, breve": {"texto": "whale ocean", "intencion": "Encontrar relatos de ballenas y el mar.", "df_objetivo": ["whale", "ocean"]},
    "Q3 — misterio": {"texto": "mystery murder detective", "intencion": "Buscar historias de investigación criminal.", "df_objetivo": ["mystery", "murder", "detective"]},
    "Q4 — viaje largo": {"texto": "voyage ship sea island captain", "intencion": "Encontrar narraciones de viajes marítimos.", "df_objetivo": ["voyage", "ship", "sea", "island", "captain"]},
    "Q5 — astronomía": {"texto": "moon stars planet telescope", "intencion": "Buscar observaciones y relatos astronómicos.", "df_objetivo": ["moon", "stars", "planet", "telescope"]},
    "Q6 — relaciones": {"texto": "marriage husband wife happiness", "intencion": "Encontrar textos sobre matrimonio y vida conyugal.", "df_objetivo": ["marriage", "husband", "wife", "happiness"]},
    "Q7 — términos raros": {"texto": "thaumaturgia marvellous elucidations", "intencion": "Localizar textos sobre fenómenos maravillosos.", "df_objetivo": ["thaumaturgia", "marvellous", "elucidations"]},
    "Q8 — término ausente": {"texto": "zzzxqvnotincorpus", "intencion": "Comprobar el resultado cuando no hay términos en vocabulario.", "df_objetivo": ["zzzxqvnotincorpus"]},
}

# TF-IDF con conteos absolutos e IDF ln(N/DF)+1; conserva la representación CSR dispersa.
vectorizer = TfidfVectorizer(
    tokenizer=tokenize, token_pattern=None, lowercase=False, vocabulary={t: i for i, t in enumerate(vocabulary)},
    use_idf=True, smooth_idf=False, norm='l2', sublinear_tf=False
)
X_tfidf = vectorizer.fit_transform([raw_texts[d] for d in doc_ids]).tocsr()
assert X_tfidf.shape == (N, len(vocabulary))

# Estadísticas BM25 de colección fijadas una sola vez para todas las consultas.
K1, B = 1.2, 0.75
avgdl = float(np.mean(list(lengths.values())))

def bm25_term_contribution(term, doc_id, k1=K1, b=B):
    f = tf_counts[doc_id].get(term, 0)
    term_df = df.get(term, 0)
    if f == 0 or term_df == 0:
        return 0.0
    idf_bm25 = math.log(1 + (N - term_df + 0.5) / (term_df + 0.5))
    dl = lengths[doc_id]
    return idf_bm25 * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avgdl))

def bm25_score(query_tokens, doc_id, k1=K1, b=B):
    # Consulta por bolsa de palabras: la frecuencia de repetición en la consulta escala el aporte.
    return sum(qtf * bm25_term_contribution(term, doc_id, k1, b)
               for term, qtf in Counter(query_tokens).items())

def rank_tfidf(query_tokens, limit=10):
    q = vectorizer.transform([" ".join(query_tokens)]).tocsr()
    scores = (X_tfidf @ q.T).toarray().ravel()
    ranked = sorted(((doc_ids[i], float(scores[i])) for i in range(N) if scores[i] > 0),
                    key=lambda item: (-item[1], str(item[0])))
    return ranked[:limit]

def rank_bm25(query_tokens, k1=K1, b=B, limit=10):
    ranked = [(d, bm25_score(query_tokens, d, k1, b)) for d in doc_ids]
    return sorted(((d, s) for d, s in ranked if s > 0), key=lambda item: (-item[1], str(item[0])))[:limit]

resultados_tfidf, resultados_bm25 = {}, {}
for nombre, q in consultas.items():
    qt = tokenize(q["texto"])
    resultados_tfidf[nombre] = rank_tfidf(qt)
    resultados_bm25[nombre] = rank_bm25(qt)
    print(f"{nombre}: {q['texto']!r} — {q['intencion']}")
    print("  DF de términos:", {t: df.get(t, 0) for t in qt})
    print("  TF-IDF:", resultados_tfidf[nombre][:3])
    print("  BM25:", resultados_bm25[nombre][:3])
print(f"\nTF-IDF dispersa: {X_tfidf.shape[0]} × {X_tfidf.shape[1]}, nnz={X_tfidf.nnz:,}; k1={K1}, b={B}, avgdl={avgdl:,.1f}")

Q1 — comunes, breve: 'love time' — Buscar pasajes sobre amor y tiempo.
  DF de términos: {'love': 85, 'time': 95}
  TF-IDF: [('pg10063.txt', 0.05557077592999217), ('pg10006.txt', 0.04411542876907718), ('pg10039.txt', 0.03687163393721664)]
  BM25: [('pg10063.txt', 0.48597242758388814), ('pg10039.txt', 0.48457746981642713), ('pg10064.txt', 0.4845374021164596)]
Q2 — ballenas, breve: 'whale ocean' — Encontrar relatos de ballenas y el mar.
  DF de términos: {'whale': 12, 'ocean': 59}
  TF-IDF: [('pg10015.txt', 0.02233057092522127), ('pg10040.txt', 0.015988048631668257), ('pg10060.txt', 0.0025069220482569223)]
  BM25: [('pg10015.txt', 5.1344436265363065), ('pg10040.txt', 4.075821929611335), ('pg10009.txt', 3.886080076420423)]
Q3 — misterio: 'mystery murder detective' — Buscar historias de investigación criminal.
  DF de términos: {'mystery': 62, 'murder': 47, 'detective': 16}
  TF-IDF: [('pg10082.txt', 0.02496500559587413), ('pg10059.txt', 0.012925454606084442), ('pg10067.txt', 0.01093099893

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
# 3. Comparación de listas, coincidencia del top 10 y diagnóstico por término.
resumen_consultas = {}
for nombre, q in consultas.items():
    a, b = resultados_tfidf[nombre], resultados_bm25[nombre]
    ids_a, ids_b = [d for d, _ in a], [d for d, _ in b]
    resumen_consultas[nombre] = {"tfidf": a, "bm25": b}
    print(f"\n{nombre} — {q['texto']}")
    print(f"IDs compartidos en top 10: {len(set(ids_a) & set(ids_b))}; tamaños: TF-IDF={len(a)}, BM25={len(b)}")
    print("Primer documento TF-IDF:", (ids_a[0], titles[ids_a[0]], a[0][1]) if a else "sin resultados")
    print("Primer documento BM25:", (ids_b[0], titles[ids_b[0]], b[0][1]) if b else "sin resultados")
    filas = []
    for pos in range(10):
        da = a[pos][0] if pos < len(a) else None
        db = b[pos][0] if pos < len(b) else None
        filas.append({
            "Posición": pos + 1,
            "ID y título TF-IDF": f"{da} — {titles[da]}" if da else "",
            "Coseno": a[pos][1] if pos < len(a) else None,
            "ID y título BM25": f"{db} — {titles[db]}" if db else "",
            "Score BM25": b[pos][1] if pos < len(b) else None,
        })
    display(pd.DataFrame(filas))

# Elegimos Q3 y dos documentos cuyo puesto cambie; si fuera necesario, buscamos el par con mayor diferencia.
analisis_q = "Q3 — misterio"
a, b = resultados_tfidf[analisis_q], resultados_bm25[analisis_q]
rank_a = {d: i + 1 for i, (d, _) in enumerate(a)}
rank_b = {d: i + 1 for i, (d, _) in enumerate(b)}
comunes = set(rank_a) & set(rank_b)
if len(comunes) >= 2:
    elegidos = sorted(comunes, key=lambda d: (-(abs(rank_a[d] - rank_b[d])), d))[:2]
else:
    elegidos = list(dict.fromkeys([d for d, _ in a[:2]] + [d for d, _ in b[:2]]))[:2]
qt = tokenize(consultas[analisis_q]["texto"])
print(f"\nAportes por término para {analisis_q}: {consultas[analisis_q]['texto']!r}")
filas = []
for d in elegidos:
    for term in dict.fromkeys(qt):
        filas.append({"Documento": d, "Título": titles[d], "Puesto TF-IDF": rank_a.get(d),
                      "Coseno": next((s for x, s in a if x == d), 0.0),
                      "Puesto BM25": rank_b.get(d), "Score BM25": next((s for x, s in b if x == d), 0.0),
                      "|d|": lengths[d], "término": term, "f(t,d)": tf_counts[d].get(term, 0),
                      "df(t)": df.get(term, 0), "aporte BM25": bm25_term_contribution(term, d)})
display(pd.DataFrame(filas))
print("El coseno usa el producto punto entre vectores TF-IDF L2 normalizados; BM25 suma aportes saturados, ponderados por IDF y normalizados por longitud.")


Q1 — comunes, breve — love time
IDs compartidos en top 10: 8; tamaños: TF-IDF=10, BM25=10
Primer documento TF-IDF: ('pg10063.txt', 'Happiness and Marriage', 0.05557077592999217)
Primer documento BM25: ('pg10063.txt', 'Happiness and Marriage', 0.48597242758388814)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg10063.txt — Happiness and Marriage,0.055571,pg10063.txt — Happiness and Marriage,0.485972
1,2,pg10006.txt — La Fiammetta,0.044115,"pg10039.txt — The Works of Aphra Behn, Volume III",0.484577
2,3,"pg10039.txt — The Works of Aphra Behn, Volume III",0.036872,pg10064.txt — Beltane the Smith,0.484537
3,4,"pg1003.txt — Divine Comedy, Longfellow's Trans...",0.026825,pg10031.txt — The Complete Poetical Works of E...,0.484184
4,5,"pg1007.txt — Divine Comedy, Cary's Translation...",0.026664,pg10006.txt — La Fiammetta,0.483907
5,6,pg10075.txt — A Narrative of the Life and Adve...,0.026422,pg10041.txt — The Rivet in Grandfather's Neck:...,0.483379
6,7,pg10031.txt — The Complete Poetical Works of E...,0.025973,pg10085.txt — Moorish Literature,0.483292
7,8,pg10085.txt — Moorish Literature,0.024817,"pg1004.txt — Divine Comedy, Longfellow's Trans...",0.483192
8,9,pg10037.txt — A Beautiful Possibility,0.023680,pg10037.txt — A Beautiful Possibility,0.483185
9,10,pg10041.txt — The Rivet in Grandfather's Neck:...,0.023394,"pg1003.txt — Divine Comedy, Longfellow's Trans...",0.483134



Q2 — ballenas, breve — whale ocean
IDs compartidos en top 10: 7; tamaños: TF-IDF=10, BM25=10
Primer documento TF-IDF: ('pg10015.txt', 'Punchinello, Volume 1, No. 19, August 6, 1870', 0.02233057092522127)
Primer documento BM25: ('pg10015.txt', 'Punchinello, Volume 1, No. 19, August 6, 1870', 5.1344436265363065)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,"pg10015.txt — Punchinello, Volume 1, No. 19, A...",0.022331,"pg10015.txt — Punchinello, Volume 1, No. 19, A...",5.134444
1,2,pg10040.txt — Aleutian Indian and English Dict...,0.015988,pg10040.txt — Aleutian Indian and English Dict...,4.075822
2,3,pg10060.txt — Discourses: Biological & Geological,0.002507,"pg10009.txt — Wild Northern Scenes; Or, Sporti...",3.886080
3,4,pg10075.txt — A Narrative of the Life and Adve...,0.002178,"pg10079.txt — The Atlantic Monthly, Volume 02,...",3.449583
4,5,pg10005.txt — A Voyage to the Moon,0.002026,"pg1005.txt — Divine Comedy, Cary's Translation...",3.362113
5,6,"pg1005.txt — Divine Comedy, Cary's Translation...",0.001788,pg10041.txt — The Rivet in Grandfather's Neck:...,3.073867
6,7,"pg10036.txt — Punchinello, Volume 2, No. 28, O...",0.001773,pg10075.txt — A Narrative of the Life and Adve...,3.046798
7,8,"pg10018.txt — Punchinello, Volume 1, No. 05, ...",0.001729,"pg10020.txt — The Strand Magazine: Vol. 07, Is...",2.980392
8,9,"pg10009.txt — Wild Northern Scenes; Or, Sporti...",0.001423,"pg10018.txt — Punchinello, Volume 1, No. 05, ...",2.935085
9,10,pg10089.txt — Elves and Heroes,0.001313,"pg10036.txt — Punchinello, Volume 2, No. 28, O...",2.931163



Q3 — misterio — mystery murder detective
IDs compartidos en top 10: 8; tamaños: TF-IDF=10, BM25=10
Primer documento TF-IDF: ('pg10082.txt', 'The Hampstead Mystery', 0.02496500559587413)
Primer documento BM25: ('pg10082.txt', 'The Hampstead Mystery', 6.4803778729535235)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg10082.txt — The Hampstead Mystery,0.024965,pg10082.txt — The Hampstead Mystery,6.480378
1,2,pg10059.txt — Aunt Jane's Nieces on Vacation,0.012925,pg10083.txt — The House of the Whispering Pines,6.179536
2,3,pg10067.txt — The Mystery of the Boule Cabinet...,0.010931,pg10029.txt — The Hunt Ball Mystery,6.087960
3,4,pg10029.txt — The Hunt Ball Mystery,0.010856,pg10059.txt — Aunt Jane's Nieces on Vacation,6.073219
4,5,pg10083.txt — The House of the Whispering Pines,0.009503,pg10067.txt — The Mystery of the Boule Cabinet...,5.915985
5,6,"pg10035.txt — Punchinello, Volume 2, No. 27, O...",0.006759,"pg10035.txt — Punchinello, Volume 2, No. 27, O...",5.581325
6,7,"pg10019.txt — Punchinello, Volume 1, No. 22, A...",0.005960,"pg10019.txt — Punchinello, Volume 1, No. 22, A...",4.969797
7,8,"pg10017.txt — Punchinello, Volume 1, No. 23, ...",0.005628,"pg10017.txt — Punchinello, Volume 1, No. 23, ...",4.957127
8,9,"pg10033.txt — Punchinello, Volume 1, No. 25, S...",0.002899,pg10008.txt — The Mystery,4.640425
9,10,"pg10016.txt — Punchinello, Volume 1, No. 21, A...",0.002883,"pg10020.txt — The Strand Magazine: Vol. 07, Is...",4.635629


... (9 outputs left out)

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,,None,,None
1,2,,None,,None
2,3,,None,,None
3,4,,None,,None
4,5,,None,,None
5,6,,None,,None
6,7,,None,,None
7,8,,None,,None
8,9,,None,,None
9,10,,None,,None



Aportes por término para Q3 — misterio: 'mystery murder detective'


,Documento,Título,Puesto TF-IDF,Coseno,Puesto BM25,Score BM25,|d|,término,"f(t,d)",df(t),aporte BM25
0,pg10083.txt,The House of the Whispering Pines,5,0.009503,2,6.179536,115520,mystery,10,62,0.870201
1,pg10083.txt,The House of the Whispering Pines,5,0.009503,2,6.179536,115520,murder,33,47,1.558856
2,pg10083.txt,The House of the Whispering Pines,5,0.009503,2,6.179536,115520,detective,34,16,3.750479
3,pg10059.txt,Aunt Jane's Nieces on Vacation,2,0.012925,4,6.073219,52048,mystery,11,62,0.957826
4,pg10059.txt,Aunt Jane's Nieces on Vacation,2,0.012925,4,6.073219,52048,murder,4,47,1.295016
5,pg10059.txt,Aunt Jane's Nieces on Vacation,2,0.012925,4,6.073219,52048,detective,26,16,3.820377


El coseno usa el producto punto entre vectores TF-IDF L2 normalizados; BM25 suma aportes saturados, ponderados por IDF y normalizados por longitud.


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [4]:
# 4. Efecto de k1 y b, manteniendo fijas las estadísticas del corpus.
consulta_param = "Q3 — misterio"
qt = tokenize(consultas[consulta_param]["texto"])
configuraciones = [("Base", 1.2, 0.75), ("Sin ajuste por longitud", 1.2, 0.0), ("Mayor efecto de repeticiones", 2.0, 0.75)]
rankings_param = {nombre: rank_bm25(qt, k1=k1, b=b, limit=N) for nombre, k1, b in configuraciones}
# Mismos documentos para observar cómo cambian sus posiciones; se toman los top 3 de la base y documentos de longitudes extremas que coinciden.
base = rankings_param["Base"]
seleccion = [d for d, _ in base[:3]]
coincidentes = [d for d in doc_ids if any(tf_counts[d].get(t, 0) for t in set(qt))]
if coincidentes:
    cortos = sorted(coincidentes, key=lambda d: (lengths[d], d))
    largos = sorted(coincidentes, key=lambda d: (-lengths[d], d))
    for d in (cortos[0], largos[0]):
        if d not in seleccion:
            seleccion.append(d)

posiciones = {nombre: {d: i + 1 for i, (d, _) in enumerate(ranking)} for nombre, ranking in rankings_param.items()}
scores_param = {nombre: dict(ranking) for nombre, ranking in rankings_param.items()}
filas = []
for d in seleccion:
    fila = {"Documento": d, "Título": titles[d], "|d|": lengths[d], "Longitud / avgdl": lengths[d] / avgdl}
    for nombre, _, _ in configuraciones:
        fila[f"Puesto {nombre}"] = posiciones[nombre].get(d, "—")
        fila[f"Score {nombre}"] = scores_param[nombre].get(d, 0.0)
    filas.append(fila)
display(pd.DataFrame(filas))

aportes = []
for d in seleccion:
    for term in dict.fromkeys(qt):
        fila = {"Documento": d, "Término": term, "f(t,d)": tf_counts[d].get(term, 0), "df(t)": df.get(term, 0)}
        for nombre, k1, b in configuraciones:
            fila[f"Aporte {nombre}"] = bm25_term_contribution(term, d, k1, b)
        aportes.append(fila)
display(pd.DataFrame(aportes))
print("b=0 elimina la penalización/compensación por longitud; b=0.75 ajusta según |d|/avgdl. k1 mayor retrasa la saturación de TF, así que las repeticiones adicionales conservan más efecto. Las posiciones dependen además de los otros términos de consulta y de los documentos competidores.")

,Documento,Título,|d|,Longitud / avgdl,Puesto Base,Score Base,Puesto Sin ajuste por longitud,Score Sin ajuste por longitud,Puesto Mayor efecto de repeticiones,Score Mayor efecto de repeticiones
0,pg10082.txt,The Hampstead Mystery,112318,1.981252,1,6.480378,1,6.571958,1,8.648580
1,pg10083.txt,The House of the Whispering Pines,115520,2.037734,2,6.179536,2,6.394183,2,8.025660
2,pg10029.txt,The Hunt Ball Mystery,54933,0.969000,3,6.087960,3,6.075277,3,7.852442
3,pg10001.txt,Apocolocyntosis,8597,0.151648,41,1.155363,55,0.754391,41,1.310106
4,pg10064.txt,Beltane the Smith,180935,3.191633,26,1.868662,18,2.318160,26,2.109570


,Documento,Término,"f(t,d)",df(t),Aporte Base,Aporte Sin ajuste por longitud,Aporte Mayor efecto de repeticiones
0,pg10082.txt,mystery,39,62,1.002359,1.024379,1.322160
1,pg10082.txt,murder,204,47,1.642884,1.649954,2.225300
2,pg10082.txt,detective,53,16,3.835135,3.897624,5.101121
3,pg10083.txt,mystery,10,62,0.870201,0.942767,1.062111
4,pg10083.txt,murder,33,47,1.558856,1.601426,2.042988
5,pg10083.txt,detective,34,16,3.750479,3.849990,4.920561
6,pg10029.txt,mystery,46,62,1.029662,1.029054,1.381206
7,pg10029.txt,murder,5,47,1.344486,1.338435,1.627362
8,pg10029.txt,detective,16,16,3.713812,3.707788,4.843874
9,pg10001.txt,mystery,0,62,0.000000,0.000000,0.000000


b=0 elimina la penalización/compensación por longitud; b=0.75 ajusta según |d|/avgdl. k1 mayor retrasa la saturación de TF, así que las repeticiones adicionales conservan más efecto. Las posiciones dependen además de los otros términos de consulta y de los documentos competidores.
